# 032 — CGC ESI Controller Test (firmware/DLL 1-00)

Real-hardware gate for the P9 session-1 class rework (esibd_bs `15cb9ff`).
Exercises the reworked `ESI` class against firmware 1-00: **configuration
selection, HV voltages (modules 2 + 3), heater temperature, measurement
ranges** — plus the open questions from the rework.

**Operator workflow (CGC lead engineer, email 2026-07-21):** load an NVM
configuration (sets heater temperature, interlocks, module enables), *then*
adjust the HV target voltages.

Device memory ships `COM-ESI-CTRL-2xHVPS.cfg`:

| Slots | Name | Notes |
|---|---|---|
| 1 | Off | everything disabled |
| 2 | Standby | DeviceEnable=N, HV modules enabled, heater limit **180 W** |
| 10–25 | Heat 30…175deg | DeviceEnable=Y, heater limit 20 W (30/40 °C slots) |
| 100–180 | HV1 +0…+3000V / −0…−3000V | HVPS1Enable=Y only |
| 200–280 | HV2 presets | analogous |

**Open questions this notebook answers on hardware:**
1. What flips `ST_ON` ↔ `ST_STBY`? (assumed: `DeviceEnable` of the loaded
   config; also test `set_enable()` alone)
2. Preset mapping: cfg keys `HVPS1..3` vs lab module addresses **2** (inlet)
   and **3** (emitter) — which address does "HV1 +100V" hit?
3. Do the two 1-00 signature fixes hold on hardware
   (`get_base_housekeeping` leading `Valid`, `get_complete_state` 9 args)?

**Safety**
- ⚠️ SINGLE-INSTANCE DLL: never run the Explorer ESI plugin and this
  notebook at the same time (second connect fails loudly by design).
- ⚠️ Heater: keep the power limit at **10–30 W below 50 °C** (overshoot).
  Slot 2 "Standby" leaves 180 W configured — the manual-heater cell sets
  20 W first.
- Rule: `# CHANGE ESI:` comment before every hardware-parameter block.


## Setup

In [1]:
import os
import time
import logging
from pathlib import Path
from datetime import datetime

from devices.cgc.esi.esi import ESI

repo_root = Path(os.getcwd()).parent.parent
log_dir = repo_root / "debugging" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = log_dir / f"032_cgc_esi_100_test_{timestamp}.log"

logger = logging.getLogger(f"032_cgc_esi_100_test_{timestamp}")
logger.setLevel(logging.DEBUG)

file_handler = logging.FileHandler(log_file)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(levelname)s - %(message)s"))
console_handler.setLevel(logging.DEBUG)
logger.addHandler(console_handler)

print(f"Logging to {log_file}")

Logging to C:\Users\ESIBDlab\Desktop\LAB_Code\esibd_bs\debugging\logs\032_cgc_esi_100_test_20260723_142034.log


In [2]:
COM_PORT = 14  # lab_config.toml [com_ports] ESI = 14

esi = ESI(device_id="esi_100_test", com=COM_PORT, logger=logger)

INFO - esi_100_test  COM14  initialized (230400 baud)


In [3]:
# CHANGE ESI: connect — bring-up runs open_port -> set_comspeed(230400) -> set_enable(True)
esi.connect()

INFO - esi_100_test  COM14  comspeed set to 230400
INFO - esi_100_test  COM14  setting device enable to True
INFO - esi_100_test  COM14  connected (vendor DLL, 230400 baud)


True

## Identity & modules

Firmware/DLL versions and the module map. Expected: address 0 = heat
controller HTCTRL-24-10 (`0xDB1C`, **new** — now installed), addresses
2 + 3 = HVPS-3kB (`0x0A0D`), address 1 empty.

In [4]:
print("DLL sw version :", esi.get_sw_version())
print("fw version     :", esi.get_fw_version())
print("fw date        :", esi.get_fw_date())
print("product id     :", esi.get_product_id())
print("product no     :", esi.get_product_no())
print("hw type/version:", esi.get_hw_type(), esi.get_hw_version())
print("uptime         :", esi.get_uptime())

DLL sw version : 256
fw version     : (0, 256)
fw date        : (0, 'Jul 13 2026')
product id     : (0, 'ESI Controller')
product no     : (0, 124701)
hw type/version: (0, 4333632) (0, 256)
uptime         : (0, 415.1259765625, 4320725.4951171875)


In [5]:
status, valid, max_module, presence = esi.get_module_presence()
print(f"presence status={status} valid={valid} max_module={max_module}")
labels = {esi.MODULE_NOT_FOUND: "not found", esi.MODULE_PRESENT: "PRESENT",
          esi.MODULE_INVALID: "invalid"}
for addr in range(esi.MODULE_NUM):
    line = f"  addr {addr}: {labels.get(presence[addr], presence[addr])}"
    if presence[addr] == esi.MODULE_PRESENT:
        st, dev_type = esi.get_module_dev_type(addr)
        kind = {esi.MODULE_HTCTRL_TYPE: "HTCTRL", esi.MODULE_HVPS_TYPE: "HVPS",
                esi.MODULE_BASE_TYPE: "BASE"}.get(dev_type, hex(dev_type))
        line += f"  type={kind} ({hex(dev_type)})  fw={esi.get_module_fw_version(addr)}"
    print(line)
print(f"  base module: {labels.get(presence[esi.PRESENCE_BASE], '?')}")

presence status=0 valid=True max_module=4
  addr 0: PRESENT  type=HTCTRL (0xdb1c)  fw=(0, 256)
  addr 1: not found
  addr 2: PRESENT  type=HVPS (0xa0d)  fw=(0, 256)
  addr 3: PRESENT  type=HVPS (0xa0d)  fw=(0, 256)
  base module: PRESENT


## Baseline state reads

Includes the two 1-00 signature changes: `get_base_housekeeping` now has a
leading `Valid`, `get_complete_state` lost the trailing heat-interlock arg
(9 args). Garbage values / status ≠ 0 here would mean the ctypes
signatures are wrong → **stop**.

In [6]:
print("main state     :", esi.get_main_state())
print("enable         :", esi.get_enable())
print("device state   :", esi.get_device_state())
print("voltage state  :", esi.get_voltage_state())
print("temp state     :", esi.get_temperature_state())
print("interlock state:", esi.get_interlock_state())
print("interlock enab :", esi.get_interlock_enable())  # lab default Y,N,Y,Y -> 0b1101 = 13
print("fan state      :", esi.get_fan_state())

main state     : (0, '0x15', 'STATE_ERR_ILOCK')
enable         : (0, True)
device state   : (0, '0x1', ['DS_ILOCK_FAIL'])
voltage state  : (0, '0x37', ['VS_3V3_OK', 'VS_5V0_OK', 'VS_24V_OK', 'VS_LINE_OK', 'VS_PSU_OK'])
temp state     : (0, '0x0', [])
interlock state: (0, '0xf00c', ['IS_CTRL_ILOCK_FP', 'IS_CTRL_ILOCK_RP', 'IS_CTRL_ILOCK_FP_CURR', 'IS_CTRL_ILOCK_RP_CURR', 'IS_CTRL_ILOCK_FP_LAST', 'IS_CTRL_ILOCK_RP_LAST'])
interlock enab : (0, 7)
fan state      : (0, '0xf', ['FS_FAN_OK', 'FS_FAN_SW_CURR', 'FS_FAN_SW_LAST', 'FS_FAN_ENB'])


In [7]:
# NEW 1-00 signature: (status, valid, volt_3v3, temp_cpu)
print("base hk        :", esi.get_base_housekeeping())
# (status, v24, v5, v3, temp_cpu, temp_psu)
print("housekeeping   :", esi.get_housekeeping())
print("heat-ctrl hk   :", esi.get_heat_ctrl_housekeeping())
for addr in (2, 3):
    print(f"HV{addr} hk        :", esi.get_hv_supply_housekeeping(addr))

base hk        : (0, True, 3.2981530343007917, 25.3)
housekeeping   : (0, 24.112, 5.009, 3.304, 24.55, 25.39)
heat-ctrl hk   : (0, True, 3.3003300330033003, 24.78, 4.99, 24.044, 23.34)
HV2 hk        : (0, True, 3.2948929159802307, 26.2, 4.985, 24.036, -21.53861514652289, 17.941, -17.753482102740755, 1.504, 0.024961309969547207)
HV3 hk        : (0, True, 3.2959789057350033, 25.7, 4.986, 24.034, -21.63959968838018, 17.941, -13.486546413375683, 1.496, 0.01897684824514093)


In [8]:
# NEW 1-00 shape: 9 args, no trailing HeatCtrlInterlockState
(status, data_flags, dev_state, volt_state, temp_state, fan_state,
 ilock_state, state, mod_data_flags, mod_state) = esi.get_complete_state()
print(f"status={status} state={hex(state)} dev={hex(dev_state)} "
      f"volt={hex(volt_state)} temp={hex(temp_state)} fan={hex(fan_state)} "
      f"ilock={hex(ilock_state)}")
for addr, ms in enumerate(mod_state):
    active = []
    if ms & esi.MS_CTRL_ACT:
        active.append("CTRL_ACT")
    if ms & esi.MS_MOD_ACT:
        active.append("MOD_ACT")
    if ms & esi.MS_DEV_ACT:
        active.append("DEV_ACT")
    print(f"  module[{addr}] state={hex(ms)} {' '.join(active)}")

status=0 state=0x0 dev=0x0 volt=0x37 temp=0x0 fan=0xf ilock=0xf00c
  module[0] state=0x8000 DEV_ACT
  module[1] state=0x0 
  module[2] state=0x8000 DEV_ACT
  module[3] state=0x8000 DEV_ACT
  module[4] state=0x8000 DEV_ACT


## Configuration management (new in 1-00)

Expected: `get_config_values` → (0, 1023, 53, 202); 42 active slots
matching `COM-ESI-CTRL-2xHVPS.cfg` (1, 2, 10–25, 100–180, 200–280).

In [9]:
print("config values  :", esi.get_config_values())  # (status, max_no, data_size, name_size)

status, active_slots, valid_slots = esi.list_configs()
print(f"status={status}  {len(active_slots)} active, {len(valid_slots)} valid")
for n in active_slots:
    _, name = esi.get_config_name(n)
    print(f"  slot {n:4d}  {name!r}")

config values  : (0, 1023, 53, 202)
status=0  42 active, 42 valid
  slot    0  'Off'
  slot    1  'Standby'
  slot    9  'Heat 30deg'
  slot   10  'Heat 40deg'
  slot   11  'Heat 50deg'
  slot   12  'Heat 60deg'
  slot   13  'Heat 70deg'
  slot   14  'Heat 80deg'
  slot   15  'Heat 90deg'
  slot   16  'Heat 100deg'
  slot   17  'Heat 110deg'
  slot   18  'Heat 120deg'
  slot   19  'Heat 130deg'
  slot   20  'Heat 140deg'
  slot   21  'Heat 150deg'
  slot   22  'Heat 160deg'
  slot   23  'Heat 170deg'
  slot   24  'Heat 175deg'
  slot   99  'HV1 +0V'
  slot  100  'HV1 +100V'
  slot  104  'HV1 +500V'
  slot  109  'HV1 +1000V'
  slot  119  'HV1 +2000V'
  slot  129  'HV1 +3000V'
  slot  149  'HV1 -0V'
  slot  150  'HV1 -100V'
  slot  154  'HV1 -500V'
  slot  159  'HV1 -1000V'
  slot  169  'HV1 -2000V'
  slot  179  'HV1 -3000V'
  slot  199  'HV2 +0V'
  slot  200  'HV2 +100V'
  slot  204  'HV2 +500V'
  slot  209  'HV2 +1000V'
  slot  219  'HV2 +2000V'
  slot  229  'HV2 +3000V'
  slot  249  '

In [10]:
# Current working config as raw 53-byte blob (read-only peek)
status, blob = esi.get_current_config()
print(f"status={status}  len={len(blob)}")
print(blob.hex(" "))

status=0  len=53
01 00 00 ec 53 c7 2d d6 8e 02 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 d0 07 00 00 00 00 00 00 00 00 00 00 d0 07 00 00 00 00 00 00


## ST_ON ↔ ST_STBY experiment — KEY QUESTION

1-00 removed the device-level activation state; the main state now reports
`STATE_ON (0)` vs `STATE_STBY (1)` and the hk channel `Activated` derives
from it. **Assumption to verify: `DeviceEnable` of the loaded config drives
the flip.** Slot 2 "Standby" has DeviceEnable=N, slot 10 "Heat 30deg" has
DeviceEnable=Y. Also test whether `set_enable()` alone flips it.

In [12]:
# CHANGE ESI: load NVM config 2 "Standby" (DeviceEnable=N, HV modules enabled, heater 0 degC)
print("load           :", esi.load_current_config(1))
time.sleep(1.0)
print("main state     :", esi.get_main_state())   # expect STATE_STBY?
print("enable         :", esi.get_enable())
for addr in (2, 3):
    print(f"module {addr} active:", esi.get_module_activation_state(addr))

INFO - esi_100_test  COM14  loading NVM config 1


load           : 0
main state     : (0, '0x15', 'STATE_ERR_ILOCK')
enable         : (0, False)
module 2 active: (0, True)
module 3 active: (0, True)


In [16]:
# CHANGE ESI: load NVM config 10 "Heat 30deg" (DeviceEnable=Y, heater 30 degC @ 20 W limit)
print("load           :", esi.load_current_config(10))
time.sleep(1.0)
print("main state     :", esi.get_main_state())   # expect STATE_ON?
print("enable         :", esi.get_enable())
print("heater target  :", esi.get_heat_ctrl_heater_temperature())
for addr in (2, 3):
    print(f"module {addr} active:", esi.get_module_activation_state(addr))

INFO - esi_100_test  COM14  loading NVM config 10


load           : 0
main state     : (0, '0x15', 'STATE_ERR_ILOCK')
enable         : (0, True)
heater target  : (0, 40.0)
module 2 active: (0, True)
module 3 active: (0, True)


In [17]:
# CHANGE ESI: set_enable(False) while config 10 is loaded — does enable alone flip ON->STBY?
print("set_enable(F)  :", esi.set_enable(False))
time.sleep(1.0)
print("main state     :", esi.get_main_state())
print("enable         :", esi.get_enable())

# CHANGE ESI: set_enable(True) — restore
print("set_enable(T)  :", esi.set_enable(True))
time.sleep(1.0)
print("main state     :", esi.get_main_state())

INFO - esi_100_test  COM14  setting device enable to False


set_enable(F)  : 0


INFO - esi_100_test  COM14  setting device enable to True


main state     : (0, '0x15', 'STATE_ERR_ILOCK')
enable         : (0, False)
set_enable(T)  : 0
main state     : (0, '0x15', 'STATE_ERR_ILOCK')


In [15]:
esi.list_configs()

(0,
 [0,
  1,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  99,
  100,
  104,
  109,
  119,
  129,
  149,
  150,
  154,
  159,
  169,
  179,
  199,
  200,
  204,
  209,
  219,
  229,
  249,
  250,
  254,
  259,
  269,
  279],
 [0,
  1,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  99,
  100,
  104,
  109,
  119,
  129,
  149,
  150,
  154,
  159,
  169,
  179,
  199,
  200,
  204,
  209,
  219,
  229,
  249,
  250,
  254,
  259,
  269,
  279])

In [23]:
# CHANGE ESI: back to NVM config 2 "Standby"
print("load           :", esi.load_current_config())
time.sleep(1.0)
print("main state     :", esi.get_main_state())

INFO - esi_100_test  COM14  loading NVM config 9


load           : 0
main state     : (0, '0x15', 'STATE_ERR_ILOCK')


## Heater (HTCTRL-24-10, address 0)

⚠️ Power limit must stay **10–30 W below 50 °C** (overshoot; 150–180 W is
for ≥50 °C only). Slot 2 "Standby" configures 180 W — the manual-target
cell below sets **20 W** first. Negative target = temperature control off.

In [24]:
print("hw limits      :", esi.get_heat_ctrl_hw_limits())  # (status, maxV, maxA, maxW, maxT)
print("volt limit     :", esi.get_heat_ctrl_voltage_limit())
print("curr limit     :", esi.get_heat_ctrl_current_limit())
print("power limit    :", esi.get_heat_ctrl_power_limit())
print("target temp    :", esi.get_heat_ctrl_heater_temperature())
# (status, valid, volt_out, volt_heat, curr_out, temp_heat)
print("monitoring     :", esi.get_heat_ctrl_monitoring())

hw limits      : (0, 21.999616, 12.000256, 179.999931891712, 175.0)
volt limit     : (0, 21.999616)
curr limit     : (0, 12.000256)
power limit    : (0, 19.999515213824)
target temp    : (0, 30.0)
monitoring     : (0, True, 0.40907, 0.005752, 0.008918, 22.084)


In [ ]:
# CHANGE ESI: heater power limit 20 W (mandatory below 50 degC), then target 40 degC
print("set power limit:", esi.set_heat_ctrl_power_limit(20.0))
print("power limit    :", esi.get_heat_ctrl_power_limit())
print("set target     :", esi.set_heat_ctrl_heater_temperature(40.0))
print("target temp    :", esi.get_heat_ctrl_heater_temperature())

In [ ]:
# Watch the heater approach the target (read-only; ~2 min at 2 s)
t0 = time.time()
while time.time() - t0 < 120:
    status, valid, vout, vheat, iout, theat = esi.get_heat_ctrl_monitoring()
    power = vout * iout
    print(f"{time.time() - t0:5.0f} s  Temp_Heater={theat:5.1f} degC  "
          f"Vout={vout:5.2f} V  Iout={iout:5.2f} A  P={power:5.1f} W  valid={valid}")
    time.sleep(2.0)

In [ ]:
# CHANGE ESI: heater temperature control OFF (negative target)
print("set target     :", esi.set_heat_ctrl_heater_temperature(-1.0))
print("target temp    :", esi.get_heat_ctrl_heater_temperature())
print("monitoring     :", esi.get_heat_ctrl_monitoring())

## HV modules (2 = inlet, 3 = emitter)

Config first: module enables come from the loaded config; from slot 2
"Standby" both HV modules are enabled but the device sits in STBY. If
targets don't take / outputs stay 0 in STBY, load slot 10 (ON) and retry —
**record which it was**.

Meas ranges (CGC email): `NegOut` picks which output (neg/pos) is
measured + regulated (voltage present on BOTH); `HighCurrRange` N ≈ 170 µA
(~35 pArms noise), Y ≈ 1.7 mA (~200 pArms).

In [ ]:
for addr in (2, 3):
    # (status, volt_neg, curr_high)
    print(f"HV{addr} meas ranges:", esi.get_hv_supply_meas_ranges(addr))
    print(f"HV{addr} target     :", esi.get_hv_supply_target_output_voltage(addr))
    print(f"HV{addr} V readback :", esi.get_hv_supply_output_voltage(addr))
    print(f"HV{addr} I readback :", esi.get_hv_supply_output_current(addr))

In [ ]:
# CHANGE ESI: HV module 2 target +50 V (low first)
print("set target     :", esi.set_hv_supply_target_output_voltage(2, 50.0))
time.sleep(2.0)
print("target readback:", esi.get_hv_supply_target_output_voltage(2))
print("V readback     :", esi.get_hv_supply_output_voltage(2))
print("I readback     :", esi.get_hv_supply_output_current(2))

In [ ]:
# CHANGE ESI: HV module 2 target +300 V (nb-024 inlet level)
print("set target     :", esi.set_hv_supply_target_output_voltage(2, 300.0))
time.sleep(2.0)
print("target readback:", esi.get_hv_supply_target_output_voltage(2))
print("V readback     :", esi.get_hv_supply_output_voltage(2))
print("I readback     :", esi.get_hv_supply_output_current(2))

In [ ]:
# CHANGE ESI: HV module 3 target +50 V, then +300 V
print("set 50 V       :", esi.set_hv_supply_target_output_voltage(3, 50.0))
time.sleep(2.0)
print("V readback     :", esi.get_hv_supply_output_voltage(3))
print("set 300 V      :", esi.set_hv_supply_target_output_voltage(3, 300.0))
time.sleep(2.0)
print("target readback:", esi.get_hv_supply_target_output_voltage(3))
print("V readback     :", esi.get_hv_supply_output_voltage(3))
print("I readback     :", esi.get_hv_supply_output_current(3))

In [ ]:
# CHANGE ESI: none — read-only live monitor (60 s at 2 s): HV V/I + heater temp
t0 = time.time()
while time.time() - t0 < 60:
    rows = []
    for addr in (2, 3):
        _, _, volts = esi.get_hv_supply_output_voltage(addr)
        _, _, amps = esi.get_hv_supply_output_current(addr)
        rows.append(f"HV{addr}: {volts:8.2f} V {amps:10.3e} A")
    st, valid, _, _, _, theat = esi.get_heat_ctrl_monitoring()
    rows.append(f"heater: {theat:5.1f} degC" if st == esi.NO_ERR and valid
                else "heater: n/a")
    print("  |  ".join(rows))
    time.sleep(2.0)

In [ ]:
# CHANGE ESI: module 2 HighCurrRange Y (1.7 mA range), verify, then back to N
print("set ranges     :", esi.set_hv_supply_meas_ranges(2, False, True))
print("readback       :", esi.get_hv_supply_meas_ranges(2))
print("I readback     :", esi.get_hv_supply_output_current(2))  # noise ~200 pArms now?

print("restore ranges :", esi.set_hv_supply_meas_ranges(2, False, False))
print("readback       :", esi.get_hv_supply_meas_ranges(2))

In [ ]:
# CHANGE ESI: HV targets back to 0 V (both modules)
for addr in (2, 3):
    print(f"HV{addr} -> 0 V    :", esi.set_hv_supply_target_output_voltage(addr, 0.0))
time.sleep(2.0)
for addr in (2, 3):
    print(f"HV{addr} V readback:", esi.get_hv_supply_output_voltage(addr))

## HV preset mapping check

The shipped presets are named "HV1"/"HV2" and set cfg keys `HVPS1..3` —
but the lab modules sit on addresses **2** and **3**. Which address does
"HV1 +100V" (slot 101) actually drive? Read the targets on both modules
after loading it.

In [ ]:
# CHANGE ESI: load NVM config 101 "HV1 +100V" (DeviceEnable=Y) — mapping probe
print("load           :", esi.load_current_config(101))
time.sleep(1.0)
print("main state     :", esi.get_main_state())
for addr in (2, 3):
    print(f"HV{addr} target     :", esi.get_hv_supply_target_output_voltage(addr))
    print(f"HV{addr} V readback :", esi.get_hv_supply_output_voltage(addr))
    print(f"module {addr} active:", esi.get_module_activation_state(addr))

In [ ]:
# CHANGE ESI: back to NVM config 2 "Standby"
print("load           :", esi.load_current_config(2))
time.sleep(1.0)
print("main state     :", esi.get_main_state())

## Fan slide-switch override (new in 1-00)

Safe per CGC email: fan is temperature-driven (runs from ≥30 °C) and the
device deactivates on critical temperature even with the fan forced off.

In [ ]:
print("fan state      :", esi.get_fan_state())
print("fan data       :", esi.get_fan_data())  # (status, failed, max, set, meas, pwm)
print("fan override   :", esi.get_fan_switch_override())

In [ ]:
# CHANGE ESI: force fan ON via override, verify, then release override
print("override ON    :", esi.set_fan_switch_override(esi.FS_OVERRIDE | esi.FS_ON))
time.sleep(2.0)
print("fan override   :", esi.get_fan_switch_override())
print("fan data       :", esi.get_fan_data())

print("release        :", esi.set_fan_switch_override(0))
print("fan override   :", esi.get_fan_switch_override())

## Housekeeping cycle

One manual `hk_monitor()` cycle — canonical log lines for every channel,
including the 1-00 `Activated` derivation (`main_state == STATE_ON`) and
the new `Temp_Heater`. With slot 2 loaded, expect `Activated 0`.

In [ ]:
esi.hk_monitor()

## Safe shutdown

Park the device: HV targets are already 0 V; load slot 1 "Off"
(everything disabled), then disconnect.

In [ ]:
# CHANGE ESI: load NVM config 1 "Off" (device + modules + heater all disabled)
print("load           :", esi.load_current_config(1))
time.sleep(1.0)
print("main state     :", esi.get_main_state())
print("enable         :", esi.get_enable())

In [8]:
esi.disconnect()

INFO - esi_100_test  COM14  disconnected


True

## Findings (fill in on hardware)

- **ST_ON ↔ ST_STBY driver:** config `DeviceEnable`? `set_enable()`?
  → _______
- **`set_enable(False)` while ON:** state became _______ (does bring-up's
  `set_enable(True)` interfere with config semantics?)
- **Preset mapping:** slot 101 "HV1 +100V" drove address ___ (module
  enables after load: 2 = ___, 3 = ___)
- **Do HV targets take in STBY** (slot 2), or only ON? → _______
- **Signature fixes:** base-hk Valid = ___, complete-state values sane? ___
- **comspeed:** bring-up log line said actual baud = _______
- **Heater:** 40 °C reached in ___ s at 20 W; overshoot ___ °C;
  `Temp_Heater` in hk = OK?
- **Meas-range toggle:** current-noise change visible? _______
- Anything unexpected → [[cgc-esi]] quirks section.
